In [1]:
# Final regressor model - rf and xgboost 
# For each cluster we train and tune a  model that regresses classified rain event 

# Workflow:
# 0. Load saved XGB 2-class classifier models (one per cluster)
# 1. Load the complete df, format cols, compute season_num
# 2. Split df by clusters and run for loop
#
#    ── TRAINING ──
#    2a. Filter training set to true rain days (insitu_prec >= 0.1)
#    2b. Fit and tune RF + XGB regressor using RandomizedSearchCV (cv=5)
#    2c. Save trained regressor model using joblib
#
#    ── EVALUATION STAGE 1: Regressor-only error ──
#    2d. Filter test set to true rain days only
#    2e. Predict and compute RMSE, MAE, R²
#        Interpretation: regressor performance in isolation, no classifier error

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

#Paths, parameters and configurations
classifier_dir = r"..\Data\Processed\Classification_Regression\Classification_models"
train_data_path = r"..\Data\Processed\Classification_Regression\final_train_df.csv"
test_data_path = r"..\Data\Processed\Classification_Regression\final_test_df.csv"
reg_models_dir =  r"..\Data\Processed\Classification_Regression\Regression_models"
reg_metrics_dir = r"..\Data\Processed\Classification_Regression\Regression_metrics"

os.makedirs(reg_metrics_dir, exist_ok= True)
os.makedirs(reg_models_dir, exist_ok= True)

random_state_num = 64
n_cores = 1
rain_threshold = 0.1
target_variable = 'insitu_prec'


In [3]:
# 1. Load and format df

train_df = pd.read_csv(train_data_path)
train_df = train_df[train_df['insitu_prec'] > rain_threshold].copy()
test_df = pd.read_csv(test_data_path)
test_df = test_df[test_df['insitu_prec'] > rain_threshold].copy()

season_dict = {'winter': 1,
                'pre_monsoon': 2, 
                'monsoon': 3,
                'post_monsoon': 4            
                }

train_df['season_num'] = train_df['season'].map(season_dict)
test_df['season_num'] = test_df['season'].map(season_dict)

train_df = train_df.dropna(subset=[target_variable])
test_df  = test_df.dropna(subset=[target_variable])
 
cluster_list = sorted(train_df['cluster'].unique().tolist())
print(f"Clusters found: {cluster_list}")


Clusters found: [0.0, 1.0, 2.0, 3.0]


In [4]:
#Helper functions

def get_param_grids():
    param_dist_rf = {
        'n_estimators':      [50, 100, 500, 1000, 2000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [50, 100, 500, 1000, 2000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    return param_dist_rf, param_dist_xgb

def compute_metrics(y_true, y_pred, label):
    """Compute RMSE, MAE, R² and print with label."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"    [{label}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  n={len(y_true)}")
    return {'rmse': round(rmse, 4), 'mae': round(mae, 4), 'r2': round(r2, 4),
            'n': len(y_true)}

In [ ]:
# 2. Split df by clusters and run for loop

cols_to_drop = [target_variable, 'season', 'date', 'cluster']
#    ── TRAINING ──
#    2a. Load classifier and filter dataframe for that cluster 
#    2b. Fit and tune RF + XGB regressor using RandomizedSearchCV (cv=5)
#    2c. Save trained regressor model using joblib
#    2d. run train and test validation 


for cluster in cluster_list:
    print(f"\n{'='*55}\nProcessing Cluster {cluster}\n{'='*55}")

    # 2a. Load classifier and filter dataframe for that cluster
    clasif_path = os.path.join(classifier_dir, f"cluster{cluster}_xgb_classifier_2classes.pkl")
    if not os.path.exists(clasif_path):
        print(f"  [SKIP] Classifier not found: {clasif_path}")
        continue
    clasif_model = joblib.load(clasif_path)

    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    feature_cols = cluster_train.drop(columns = cols_to_drop, errors = 'ignore').columns.tolist()

    # 2b. fit a regressor and tune
    X_train = cluster_train.drop(columns = cols_to_drop, errors = 'ignore').values
    y_train = cluster_train[target_variable].values

    param_dist_rf, param_dist_xgb = get_param_grids()

    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('reg',     RandomForestRegressor(random_state=random_state_num,
                                                  n_jobs=n_cores)),
            ])
            param_dist = {f'reg__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('reg',     XGBRegressor(random_state=random_state_num,
                                         n_jobs=n_cores,
                                         verbosity=0)),
            ])
            param_dist = {f'reg__{k}': v for k, v in param_dist_xgb.items()}
        search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_dist,
            n_iter=50,
            cv=5,
            verbose=1,
            random_state=random_state_num,
            n_jobs=n_cores,
            scoring='neg_root_mean_squared_error',
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_

        print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
        print(f"    Best params : {search.best_params_}")
        print(f"    CV RMSE     : {-search.best_score_:.4f}")

        #    2c. Save trained regressor model using joblib
        model_path = os.path.join(reg_models_dir, f"cluster{cluster}_{model_type}_regressor.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model {model_type} saved to {model_path}")

        #    2d. run train and test validation 
        #validation on train set
        train_metrics = compute_metrics(y_train, best_model.predict(X_train),
                                        'Train — true rain days')
        #validation on test set
        X_test = cluster_test.drop(columns = cols_to_drop, error = ignore).values
        y_test = cluster_test[target_variable].values
        y_pred = best_model.predict(X_test)
        test_metrics = compute_metrics(y_test, best_model.predict(X_test),
                                        'Test — true rain days')
        result.append({
            'cluster': cluster,
            'model': model_type.upper(),
            'Train-RMSE': train_metrics['rmse'],
            'Train-MAE': train_metrics['mae'],
            'Train-R2': train_metrics['r2'],
            'Train-n': train_metrics['n'],
            'Test-RMSE': test_metrics['rmse'],
            'Test-MAE': test_metrics['mae'],
            'Test-R2': test_metrics['r2'],
            'Test-n': test_metrics['n']            
        })

results_df = pd.DataFrame(results)

avg_rows = (
    results_df.groupby('model')[
        ['Train-RMSE', 'Train-MAE', 'Train-R2',
         'Test-RMSE', 'Test-MAE',
         'Test-R2']
    ].mean().round(4).reset_index().assign(cluster='AVERAGE')
)

summary_df = pd.concat([results_df, avg_rows], ignore_index=True)
summary_df = summary_df[['cluster', 'model',
                    'Train-RMSE', 'Train-MAE', 'Train-R2',
                    'Test-RMSE', 'Test-MAE',
                    'Test-R2']]


Processing Cluster 0.0
Fitting 5 folds for each of 50 candidates, totalling 250 fits
